# 凑出 24 ― 寻找 4 个数与运算符的组合

## 本章要点

用 Z3 解“凑出 24”的谜题。只要定下 4 个数、3 个运算符和计算顺序，答案就出来了。

- 4 个数是一个“排列”，用约束来表示重排（即使有重复的数也能正常工作）
- 中间结果会出现分数，所以变量用 `Real`（实数）
- 运算符用 0〜5 的整数表示，用 `Implies` 写“这个编号时做什么计算”

我们还会用 `Function()` 把同一个问题再写一版。

## 问题

用 4 个数和 `+`, `-`, `*`, `/` 以及括号，做出答案等于 `24` 的算式。

比如 `[1, 5, 5, 5]` 时

```
(5 - (1 / 5)) * 5 = 24
```

就是答案之一。数的排列方式有 $4! = 24$ 种（有相同的数时相应减少），运算符的选法有 $6^3$ 种（后述）。人手动去找很辛苦，所以让 Z3 来找。

## 用约束表示数的排列

`solve_24` 里想把 4 个数当作“重排后的结果”来用，所以要写排列的约束。

`permutation(xs, values)` 返回的约束让 `xs` 成为 `values` 的一个排列。

1. `xs` 的每个元素都是 `values` 中含有的值之一
2. 对每个值 `v`，它在 `xs` 中出现的次数与在 `values` 中的次数相同

`[1, 5, 5, 5]` 就是“`1` 出现 1 次、`5` 出现 3 次”。与用 `Distinct()` 的方法不同，**有多个相同的数也能正确处理**。

In [1]:
from z3 import *
from helper.z3 import all_solutions


def permutation(variables, values):
    """返回 variables 是 values 的一个排列这一约束"""
    distinct_values = set(values)
    # 1. 每个元素都是 values 中含有的值之一
    exprs = [Or([x == v for v in distinct_values]) for x in variables]
    # 2. 每个值的出现次数与 values 相同
    exprs += [Sum([x == v for x in variables]) == values.count(v) for v in distinct_values]
    return And(exprs)


xs = IntVector('x', 4)
solver = Solver()
solver.add(permutation(xs, [1, 5, 5, 5]))

for m in all_solutions(solver):
    print([m[x] for x in xs])

[5, 5, 5, 1]
[1, 5, 5, 5]
[5, 5, 1, 5]
[5, 1, 5, 5]


`[1, 5, 5, 5]` 的排列有 $4!/3! = 4$ 种，所以只输出 4 行。

## 组装算式

计算做成“从左往右一个一个算下去”的形式。

```
s0 = n0 op0 n1
s1 = s0 op1 n2
s2 = s1 op2 n3      ← 它等于 24 就行
```

这样只需要 4 个数、3 个运算符和 3 个中间结果，就能表示整个算式。

最后再从 `s2` 的值出发，把原来的算式（字符串）拼出来。

In [2]:
# 运算符编号 → 符号
# 0:+  1:-  2:*  3:/  4:反向的-  5:反向的/
OP_SYMBOLS = {0: '+', 1: '-', 2: '*', 3: '/', 4: '-', 5: '/'}


def output(numbers, ops):
    """从计算步骤（数的顺序和运算符序列）生成算式字符串"""
    def combine(n1, n2, op):
        if op in (4, 5):
            n1, n2 = n2, n1          # 反向的 - 和 / 交换左右再显示
        if isinstance(n1, str):
            n1 = f'({n1})'
        if isinstance(n2, str):
            n2 = f'({n2})'
        return f'{n1} {OP_SYMBOLS[op]} {n2}'

    expr = numbers[0]
    for n2, op in zip(numbers[1:], ops):
        expr = combine(expr, n2, op)
    return expr


print(output([1, 5, 5, 5], [3, 4, 2]))   # (5 - (1 / 5)) * 5
print(output([8, 3, 3, 8], [3, 4, 5]))   # 8 / (3 - (8 / 3))

(5 - (1 / 5)) * 5
8 / (3 - (8 / 3))


运算符编号取到 0〜5，是为了把减法、除法的**左右交换形式**也用编号表示出来。

- `-` 需要 `a - b` 和 `b - a` 两种（这样不用重排数就能都试一遍）
- `/` 同样需要 `a / b` 和 `b / a`

只有减法和除法会因顺序不同而结果不同，所以把编号分开，就不必白白增加数的重排。

## 用 Z3 求解

关键在于把 4 个数设成 `Real`（实数）。中途会出现 `8 / 3` 这样的分数，用整数就除不尽了。

`solve_24` 有解时返回算式字符串，无解时返回 `None`。

In [3]:
def solve_24(*numbers):
    solver = Solver()
    n = RealVector('n', 4)      # 重排后的 4 个数（实数）
    op = IntVector('op', 3)     # 3 个运算符（0〜5）
    acc = RealVector('s', 3)    # 中途的计算结果

    solver.add(permutation(n, numbers))

    for x in op:
        solver.add(0 <= x, x < 6)

    def calc_op(n1, o, n2, res):
        """res == n1 (o) n2 这个约束"""
        return And(
            Implies(n2 == 0, o != 3),        # 不能除以 0
            Implies(n1 == 0, o != 5),
            Implies(o == 0, res == n1 + n2),
            Implies(o == 1, res == n1 - n2),
            Implies(o == 2, res == n1 * n2),
            Implies(o == 3, res == n1 / n2),
            Implies(o == 4, res == n2 - n1),  # 左右交换的减法
            Implies(o == 5, res == n2 / n1),  # 左右交换的除法
        )

    solver.add(calc_op(n[0], op[0], n[1], acc[0]))    # s0 = n0 op0 n1
    solver.add(calc_op(acc[0], op[1], n[2], acc[1]))  # s1 = s0 op1 n2
    solver.add(calc_op(acc[1], op[2], n[3], acc[2]))  # s2 = s1 op2 n3
    solver.add(acc[2] == 24)

    if solver.check() != sat:
        return None

    m = solver.model()
    numbers_used = [m.eval(x).as_long() for x in n]
    ops_used = [m.eval(x).as_long() for x in op]
    return output(numbers_used, ops_used)

试解几道题。用 `eval()` 确认拼出来的算式是否真的等于 24。

In [4]:
questions = [
    [1, 5, 5, 5],
    [3, 3, 8, 8],
    [3, 3, 7, 7],
    [1, 4, 5, 6],
    [2, 2, 2, 9],
    [2, 7, 8, 9],
    [6, 9, 9, 10],
    [1, 2, 7, 7],
    [4, 4, 10, 10],
    [2, 5, 5, 10],
]

for q in questions:
    expr = solve_24(*q)
    if expr is None:
        print(f'{q}: 无解')
    else:
        print(f'{q}: {expr} = {eval(expr):g}')

[1, 5, 5, 5]: (5 - (1 / 5)) * 5 = 24
[3, 3, 8, 8]: 8 / (3 - (8 / 3)) = 24
[3, 3, 7, 7]: ((3 / 7) + 3) * 7 = 24


[1, 4, 5, 6]: 6 / ((5 / 4) - 1) = 24


[2, 2, 2, 9]: ((2 + 9) * 2) + 2 = 24


[2, 7, 8, 9]: ((7 + 9) * 2) - 8 = 24
[6, 9, 9, 10]: (9 / (6 / 10)) + 9 = 24
[1, 2, 7, 7]: ((7 * 7) - 1) / 2 = 24
[4, 4, 10, 10]: ((10 * 10) - 4) / 4 = 24


[2, 5, 5, 10]: (5 - (2 / 10)) * 5 = 24


每道题都解开了。Z3 返回的是**满足条件的第一个解**，所以同一个问题每次运行都可能得到不同的算式。比如 `[1, 4, 5, 6]` 无论

- `6 / ((5 / 4) - 1)`
- `4 / (1 - (5 / 6))`

都能得到 24。

## 用 Function 重写

数的排列也可以用 `Function()` 表示成“**编号 → 数**的对应表”。

```python
number_map = Function('number_map', IntSort(), RealSort())
```

这是一个“接收整数、返回实数”的函数。用 `number_map(0) == numbers[0]` 这样把编号和数绑起来。

使用时给 `indices = IntVector('i', 4)` 加上 `Distinct(indices)`，让**4 个编号互不相同**。编号不同，从 `number_map` 取出的数也就自然构成一次重排。

In [5]:
def solve_24_by_function(*numbers):
    # 编号 → 数 的对应表
    number_map = Function('number_map', IntSort(), RealSort())

    solver = Solver()
    for i, v in enumerate(numbers):
        solver.add(number_map(i) == v)

    # 取出数的顺序（4 个编号互不相同）
    indices = IntVector('i', 4)
    for x in indices:
        solver.add(0 <= x, x < 4)
    solver.add(Distinct(indices))

    op = IntVector('op', 3)
    for x in op:
        solver.add(0 <= x, x < 6)

    acc = RealVector('s', 3)

    def calc_op(n1, o, n2, res):
        return And(
            Implies(n2 == 0, o != 3),
            Implies(n1 == 0, o != 5),
            Implies(o == 0, res == n1 + n2),
            Implies(o == 1, res == n1 - n2),
            Implies(o == 2, res == n1 * n2),
            Implies(o == 3, res == n1 / n2),
            Implies(o == 4, res == n2 - n1),
            Implies(o == 5, res == n2 / n1),
        )

    n = [number_map(i) for i in indices]
    solver.add(calc_op(n[0], op[0], n[1], acc[0]))
    solver.add(calc_op(acc[0], op[1], n[2], acc[1]))
    solver.add(calc_op(acc[1], op[2], n[3], acc[2]))
    solver.add(acc[2] == 24)

    if solver.check() != sat:
        return None

    m = solver.model()
    numbers_used = [m.eval(x).as_long() for x in n]
    ops_used = [m.eval(x).as_long() for x in op]
    return output(numbers_used, ops_used)


for q in questions:
    expr = solve_24_by_function(*q)
    print(f'{q}: {expr} = {eval(expr):g}')

[1, 5, 5, 5]: (5 - (1 / 5)) * 5 = 24
[3, 3, 8, 8]: 8 / (3 - (8 / 3)) = 24
[3, 3, 7, 7]: ((3 / 7) + 3) * 7 = 24


[1, 4, 5, 6]: 6 / ((5 / 4) - 1) = 24
[2, 2, 2, 9]: ((2 + 9) * 2) + 2 = 24


[2, 7, 8, 9]: ((7 + 9) * 2) - 8 = 24
[6, 9, 9, 10]: (10 / (6 / 9)) + 9 = 24


[1, 2, 7, 7]: ((7 * 7) - 1) / 2 = 24


[4, 4, 10, 10]: ((10 * 10) - 4) / 4 = 24
[2, 5, 5, 10]: (5 - (2 / 10)) * 5 = 24


答案和前面一样。两种方法的区别只在于“怎么表示排列”。

| | 用 `permutation()` | 用 `Function` |
|---|---|---|
| 数的重排 | 用值的出现次数约束 | 编号 → 数 的对应表 + `Distinct` |
| 有多个相同的数时 | 数出现次数，自然能处理 | 编号不同即可，允许同值（自然能处理） |
| 式子的长度 | 短 | 稍长 |

两边都能解。数的种类多的题目，用对应表会更清楚。

## 小结

- 4 个数的重排可以用 `permutation()`（按出现次数约束）或 `Function` + `Distinct` 来表示
- 中间结果会出现分数，所以变量要设成 `Real`
- 运算符用 0〜5 的整数表示，用 `Implies` 把“这个编号时做什么计算”摆开
- 减法、除法把左右交换的形式也编上号，就不必增加数的重排
- 解不一定只有一个。Z3 返回的是满足条件的第一个解